In [1]:
%load_ext autoreload
%autoreload 2
import sys
import os
sys.path.append(os.path.abspath(".."))  

In [ ]:
# ## 1. Toy bilingual dataset
#
# Simple English/"unaccented Vietnamese" sentences, composed from a **subject x verb x object** grid (3x3x3 = 27 combinations) - each word appears in many different contexts so the model has a chance to learn **each word's meaning** instead of memorizing whole sentences. 4 combinations are held out as a **test set** (never shown to the model during training) to check its ability to generalize to new combinations.
#
# Set `SRC_PATH`/`TGT_PATH` below to load your own parallel corpus instead (2 plain text files, line N of one aligned with line N of the other).

import numpy as np

from models.transformer.model import Seq2SeqTransformer
from utils.loss.CrossEntropyLoss import CrossEntropyLoss
from utils.optimizers.Adam import Adam

# Set these to load your own parallel corpus instead of the toy grid below.
# Format: 2 plain text files, line N of SRC_PATH aligned with line N of
# TGT_PATH (e.g. "train.en" / "train.vi"). Leave both as None to keep
# running this notebook with the toy dataset.
SRC_PATH = None   # e.g. r"D:\data\train.en"
TGT_PATH = None   # e.g. r"D:\data\train.vi"
MAX_SENT_LEN = 40  # drop sentence pairs where either side has more words than this


def load_parallel_corpus(src_path, tgt_path, max_len=None):
    """2 aligned text files -> list of (src_sentence, tgt_sentence) pairs.
    Lowercases and whitespace-tokenizes (matches `encode_src`/`encode_tgt`
    below); swap in a real tokenizer here if your language needs one
    (e.g. no whitespace between words)."""
    with open(src_path, encoding="utf-8") as f:
        src_lines = [line.strip().lower() for line in f if line.strip()]
    with open(tgt_path, encoding="utf-8") as f:
        tgt_lines = [line.strip().lower() for line in f if line.strip()]
    if len(src_lines) != len(tgt_lines):
        raise ValueError(f"line count mismatch: {len(src_lines)} vs {len(tgt_lines)}")

    pairs = list(zip(src_lines, tgt_lines))
    if max_len is not None:
        pairs = [(s, t) for s, t in pairs
                 if len(s.split()) <= max_len and len(t.split()) <= max_len]
    return pairs


if SRC_PATH and TGT_PATH:
    train_pairs = load_parallel_corpus(SRC_PATH, TGT_PATH, max_len=MAX_SENT_LEN)
else:
    # Toy English -> "unaccented Vietnamese" bilingual data (simple whitespace
    # tokenizing): a full subject x verb x object grid, holding out 4 combos as the test set.
    subjects = [("i", "toi"), ("you", "ban"), ("we", "chung toi")]
    verbs = [("love", "yeu"), ("see", "thay"), ("hate", "ghet")]
    objects = [("cats", "meo"), ("dogs", "cho"), ("birds", "chim")]

    held_out = {("we", "love", "cats"), ("you", "love", "birds"),
                ("i", "see", "dogs"), ("we", "hate", "cats")}

    train_pairs = []
    for s_en, s_vi in subjects:
        for v_en, v_vi in verbs:
            for o_en, o_vi in objects:
                if (s_en, v_en, o_en) in held_out:
                    continue
                train_pairs.append((f"{s_en} {v_en} {o_en}", f"{s_vi} {v_vi} {o_vi}"))

PAD, BOS, EOS, UNK = 0, 1, 2, 3
SPECIALS = {"<pad>": PAD, "<bos>": BOS, "<eos>": EOS, "<unk>": UNK}


def build_vocab(sentences):
    vocab = dict(SPECIALS)
    for s in sentences:
        for w in s.split():
            if w not in vocab:
                vocab[w] = len(vocab)
    return vocab


src_sentences = [p[0] for p in train_pairs]
tgt_sentences = [p[1] for p in train_pairs]

src_vocab = build_vocab(src_sentences)
tgt_vocab = build_vocab(tgt_sentences)
tgt_id2word = {i: w for w, i in tgt_vocab.items()}

src_max_len = max(len(s.split()) for s in src_sentences)
tgt_max_words = max(len(s.split()) for s in tgt_sentences)
dec_seq_len = tgt_max_words + 1  # slot for <bos> or <eos>

print("num train sentences:", len(train_pairs))
print("src_vocab_size:", len(src_vocab), "tgt_vocab_size:", len(tgt_vocab))
print("src_max_len:", src_max_len, "dec_seq_len:", dec_seq_len)

# Reading the result: 23/27 combinations are used for training (4 combinations held out for testing), 13 unique words per side (9 content words + 4 special tokens `<pad>/<bos>/<eos>/<unk>`), the longest target sentence has 4 words ("chung toi yeu meo") so `dec_seq_len=5` (4 words + 1 slot for `<bos>`/`<eos>`).

In [3]:
# ## 2. Encode: teacher forcing
#
# During training, the Decoder doesn't generate tokens one at a time on its own (slow and hard to backpropagate through) - it uses **teacher forcing**: feeding the real target sentence directly as input, shifted 1 position from the label it must predict -
#
# - `tgt_in` (Decoder input): `<bos> w1 w2 ... wn` - at step `t` the Decoder only sees `<bos>...w(t-1)` (thanks to the causal mask in `Decoder`) to predict `wt`.
# - `tgt_out` (label for the loss): `w1 w2 ... wn <eos>` - shifted left 1 position relative to `tgt_in`, exactly the "next token" the Decoder must predict at each step.

def encode_src(sentence, vocab, max_len):
    ids = [vocab.get(w, UNK) for w in sentence.split()]
    ids = ids[:max_len] + [PAD] * max(0, max_len - len(ids))
    return ids


def encode_tgt(sentence, vocab, dec_len):
    word_ids = [vocab.get(w, UNK) for w in sentence.split()]
    tgt_in = [BOS] + word_ids
    tgt_out = word_ids + [EOS]
    tgt_in = tgt_in[:dec_len] + [PAD] * max(0, dec_len - len(tgt_in))
    tgt_out = tgt_out[:dec_len] + [PAD] * max(0, dec_len - len(tgt_out))
    return tgt_in, tgt_out


X_src = np.array([encode_src(s, src_vocab, src_max_len) for s in src_sentences])
tgt_in_out = [encode_tgt(s, tgt_vocab, dec_seq_len) for s in tgt_sentences]
X_tgt_in = np.array([t[0] for t in tgt_in_out])
X_tgt_out = np.array([t[1] for t in tgt_in_out])

print("X_src shape:", X_src.shape)
print("X_tgt_in shape:", X_tgt_in.shape, "X_tgt_out shape:", X_tgt_out.shape)
print(f'Example: "{src_sentences[0]}" -> src_ids {X_src[0]}')
print(f'        tgt_in {X_tgt_in[0]}  tgt_out {X_tgt_out[0]}')

# Reading the result: for the first sentence "i love cats" (3 words, no padding needed) -> `tgt_in = [<bos>, toi, yeu, meo, <pad>]`, `tgt_out = [toi, yeu, meo, <eos>, <pad>]` - exactly as described: `tgt_out` is `tgt_in` shifted left by 1 step, ids `4/5/6` appear in both but at shifted positions.

X_src shape: (23, 3)
X_tgt_in shape: (23, 5) X_tgt_out shape: (23, 5)
Example: "i love cats" -> src_ids [4 5 6]
        tgt_in [1 4 5 6 0]  tgt_out [4 5 6 2 0]


In [4]:
# ## 3. Training loop
#
# Combine `Seq2SeqTransformer` + `CrossEntropyLoss` (using `ignore_index=PAD` so padding positions don't contribute to the loss) + `Adam` - the same training loop pattern already used for `TextClassifierTransformer` in `week4_demo.ipynb` (`zero_grad -> forward -> loss -> backward -> step`).

np.random.seed(0)
model = Seq2SeqTransformer(
    src_vocab_size=len(src_vocab), tgt_vocab_size=len(tgt_vocab),
    d_model=32, num_heads=4, d_ff=64, num_layers=2,
    max_len=max(src_max_len, dec_seq_len), pad_id=PAD,
)
loss_fn = CrossEntropyLoss()
optimizer = Adam(model.parameters(), lr=0.01)

n_epochs = 300
for epoch in range(n_epochs):
    optimizer.zero_grad()

    probs = model(X_src, X_tgt_in)
    loss = loss_fn(probs, X_tgt_out, ignore_index=PAD)

    grad = loss_fn.backward()
    model.backward(grad)
    optimizer.step()

    if epoch % 50 == 0 or epoch == n_epochs - 1:
        print(f"Epoch {epoch:4d} - loss: {loss:.4f}")

Epoch    0 - loss: 3.4320
Epoch   50 - loss: 2.1905
Epoch  100 - loss: 0.8223
Epoch  150 - loss: 0.4514
Epoch  200 - loss: 0.3428
Epoch  250 - loss: 0.2422
Epoch  299 - loss: 0.1188


In [5]:
# ## 4. Inference on 4 NEW sentences
#
# The 4 held-out combinations (not in the training set) - using `model.generate()` so the Decoder generates one token at a time on its own (autoregressive), with no target sentence to "feed" it like during training.

test_sentences = [
    "we love cats",
    "you love birds",
    "i see dogs",
    "we hate cats",
]


def decode_ids(ids):
    words = []
    for i in ids:
        if i == EOS:
            break
        if i in (BOS, PAD):
            continue
        words.append(tgt_id2word.get(i, "<unk>"))
    return " ".join(words)


X_test = np.array([encode_src(s, src_vocab, src_max_len) for s in test_sentences])
generated = model.generate(X_test, bos_id=BOS, eos_id=EOS, max_len=dec_seq_len)

for s, ids in zip(test_sentences, generated):
    print(f'"{s}" -> "{decode_ids(ids)}"')

# Reading the result: the verb (`yeu`/`thay`/`ghet`) is translated correctly almost every time - because each verb appears in 8 different training sentences (3 subjects x 3 objects, minus the held-out ones), enough for the model to isolate its meaning. The subject/object aren't fully correct yet (e.g. `"we love cats"` comes out as `"toi yeu meo"` - missing the `chung` of `chung toi`; `"i see dogs"` comes out as `"toi thay chim"` - confusing `cho`/`chim`) because 23 sentences is still a very small set, each word only seen in a few contexts.
#
# Trying `n_epochs=600` shows the loss dropping close to 0 (the model memorizes 100% of the training set) but the result on the held-out test set is **not** better - sometimes even worse (the model starts "memorizing" training sentences instead of learning the general structure). This is a classic case of overfitting on too small a dataset: near-zero training loss doesn't mean good generalization. The repo doesn't have `Dropout` yet - this is exactly the kind of problem Dropout (and more data) exists to reduce.
#
# The main goal of this notebook is the same as the earlier demos: confirm that the whole `TokenEmbedding -> PositionalEncoding -> Encoder -> Decoder (cross-attend full sequence) -> Linear -> Softmax -> CrossEntropyLoss -> backward -> Adam` pipeline runs correctly end-to-end for a sequence-generation task, not to achieve real translation quality.

"we love cats" -> "toi yeu meo"
"you love birds" -> "ban yeu cho"
"i see dogs" -> "toi thay chim"
"we hate cats" -> "toi ghet meo"
